In [1]:
!pip install chromadb sentence-transformers transformers datasets accelerate


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 76.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.1/103.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 105.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/

In [2]:
pip install chromadb

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm
import torch
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM


2025-12-03 12:42:04.921706: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764765725.102859      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764765725.155293      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [4]:
ds = load_dataset("Amod/mental_health_counseling_conversations", split="train")
df = ds.to_pandas().head(500)
df


README.md: 0.00B [00:00, ?B/s]

combined_dataset.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/3512 [00:00<?, ? examples/s]

,Context,Response
0,I'm going through some things with my feelings...,"If everyone thinks you're worthless, then mayb..."
1,I'm going through some things with my feelings...,"Hello, and thank you for your question and see..."
2,I'm going through some things with my feelings...,First thing I'd suggest is getting the sleep y...
3,I'm going through some things with my feelings...,Therapy is essential for those that are feelin...
4,I'm going through some things with my feelings...,I first want to let you know that you are not ...
...,...,...
495,I've hit my head on walls and floors ever sinc...,The best way to handle anxiety of this level i...
496,I have several issues that I need to work thro...,I am sorry that you had this experience. Thera...
497,Why am I so afraid of it? I don't understand.,Your fear is somewhat reasonable. No one want...
498,Why am I so afraid of it? I don't understand.,Why are you afraid of rape? Because it is a pr...


In [5]:
train_df = df.sample(frac=0.7, random_state=42)
test_df = df.drop(train_df.index)

print("Train size:", len(train_df))
print("Test size:", len(test_df))


Train size: 350
Test size: 150


In [6]:
import os
if "COLAB_GPU" in os.environ:
  from google.colab import output
  output.enable_custom_widget_manager()

In [ ]:
import os
from huggingface_hub import notebook_login

# If running in Google Colab
if "COLAB_GPU" in os.environ:
    !huggingface-cli login
# If running locally (Jupyter, VS Code, etc.)
else:
    notebook_login() 


In [8]:
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embed_tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
model_name = "meta-llama/Llama-3.2-1B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16
)



tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

In [10]:
print("Tokenizer max length:", tokenizer.model_max_length)
print("Model max positions:", model.config.max_position_embeddings)

Tokenizer max length: 131072
Model max positions: 131072


In [ ]:
# CREATE CHROMA COLLECTION 

import chromadb
from chromadb.config import Settings

# Use PersistentClient for local persistence
client = chromadb.PersistentClient(path="./chroma_db03")

collection = client.create_collection(
    name="mental_health_rags",
    metadata={"hnsw:space": "cosine"}
)

In [12]:
ids = []
documents = []
metadatas = []

for i, row in train_df.iterrows():
    ctx = row["Context"]
    resp = row["Response"]

    ids.append(str(i))
    documents.append(ctx)   # vectorized text = ONLY context

    metadatas.append({
        "context": ctx,
        "actual_response": resp,
        "row_index": i
    })


In [13]:
embeddings = embed_model.encode(documents, convert_to_numpy=True)

collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas
)


In [45]:
def build_rag_prompt(user_query, retrieved_contexts, retrieved_responses):
    # Build example block
    examples = ""
    for i, (ctx, resp) in enumerate(zip(retrieved_contexts, retrieved_responses), start=1):
        examples += f"""
Example {i}
User: {ctx}
Assistant: {resp}
"""

    prompt = f"""
Below are examples of how a counselor responds with empathy and support.
Use these examples only as style guidance. 
Do NOT copy them. 
Do NOT repeat the user's message. 
Write a new, original response.

{examples}

Now the user says:
User: {user_query}

Assistant:"""
    return prompt


In [33]:
results = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="RAG Testing"):
    
    test_query = row["Context"]
    actual_test_response = row["Response"]

    # embed + retrieve
    q_emb = embed_model.encode([test_query], convert_to_numpy=True).tolist()
    retrieved = collection.query(query_embeddings=q_emb, n_results=3)

    retrieved_contexts  = retrieved["documents"][0]
    retrieved_responses = [m["actual_response"] for m in retrieved["metadatas"][0]]

    # build prompt
    prompt = build_rag_prompt(
        test_query,
        retrieved_contexts,
        retrieved_responses
    )

    # generate output
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

    gen_output = tokenizer.decode(
        out_ids[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    results.append({
        "test_index": idx,
        "user_query": test_query,
        "actual_test_response": actual_test_response,
        "retrieved_train_responses": retrieved_responses,
        "rag_generated_output": gen_output
    })


RAG Testing:   0%|          | 0/1 [00:00<?, ?it/s]

ssssssssss:v 
Below are examples of how a counselor responds with empathy and support.
Use these examples only as style guidance. 
Follow the tone and style closely.
You may reuse similar words and phrases when appropriate. 
Write a new, original response.


Example 1
User: I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.
   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.
   How can I change my feeling of being worthless to everyone?
Assistant: I'm sorry to hear you're feeling this intense emotion of worthlessness.  I'm glad to hear this has not reached the point of suicidal ideation; however, it does sounds like you could use some additional support right now.  I would recommend seeking out counseling to help you challenge the negative beliefs you have about yourself.  Although many types of therapy would be helpful, cogn

In [16]:
evaluation_df = pd.DataFrame(results)
evaluation_df


,test_index,user_query,actual_test_response,retrieved_train_responses,rag_generated_output
0,1,I'm going through some things with my feelings...,"Hello, and thank you for your question and see...",[I'm sorry to hear you're feeling this intense...,You have a lot of great questions and I'm hap...
1,4,I'm going through some things with my feelings...,I first want to let you know that you are not ...,[I'm sorry to hear you're feeling this intense...,\nOftentimes we can change our feelings about...
2,8,I'm going through some things with my feelings...,It sounds like you may be putting yourself las...,[I'm sorry to hear you're feeling this intense...,I'm sorry to hear you're feeling this intense...
3,13,I'm going through some things with my feelings...,I'm glad you are interested in changing your f...,[I'm sorry to hear you're feeling this intense...,"Heck, sure thing, hun!Feelings of 'depression..."
4,14,I'm going through some things with my feelings...,You have\nseveral things going on here. The sl...,[I'm sorry to hear you're feeling this intense...,I'm sorry to hear you're feeling this intense...
...,...,...,...,...,...
145,486,I have a fear of something and I want to face ...,This answer could be very different depending ...,"[Fear is a part of life. In fact, our five mai...","Fear is a part of life. In fact, our five mai..."
146,488,I have a fear of something and I want to face ...,"Hello, and thank you for your question. Overco...","[Fear is a part of life. In fact, our five mai...","Fear is a part of life. In fact, our five mai..."
147,492,I don't remember when the voices in my head st...,This isn't something you can do on your own. I...,[You are right. It is not normal to hear voice...,I can't tell if you are talking about the voi...
148,496,I have several issues that I need to work thro...,I am sorry that you had this experience. Thera...,[Despite your anxiety you are highly attuned t...,I understand your concern. Therapists are not...


In [ ]:
from rouge_score import rouge_scorer
import pandas as pd
import evaluate

preds = evaluation_df["rag_generated_output"].tolist()
refs  = evaluation_df["actual_test_response"].tolist()

rouge = evaluate.load("rouge")

rouge_scores = rouge.compute(
    predictions=preds,
    references=refs,
    rouge_types=["rouge1", "rouge2", "rougeL", "rougeLsum"]
)

print("ROUGE Scores:")
print(rouge_scores)



In [21]:
!pip install rouge-score bert-score -q

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.6 MB/s eta 0:00:00


In [23]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import pandas as pd


Rouge Score

In [24]:
from rouge_score import rouge_scorer
import pandas as pd
import evaluate

preds = evaluation_df["rag_generated_output"].tolist()
refs  = evaluation_df["actual_test_response"].tolist()

rouge = evaluate.load("rouge")

rouge_scores = rouge.compute(
    predictions=preds,
    references=refs,
    rouge_types=["rouge1", "rouge2", "rougeL", "rougeLsum"]
)

print("ROUGE Scores:")
print(rouge_scores)



ROUGE Scores:
{'rouge1': 0.2870736630724587, 'rouge2': 0.03971408483413606, 'rougeL': 0.1359966489115684, 'rougeLsum': 0.1502638467833938}


Bert Score

In [26]:
from tqdm.auto import tqdm
from bert_score import score as bertscore
import torch
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

bert_f1_list = []

for pred, ref in tqdm(zip(preds, refs), total=len(preds), desc="DistilBERT BERTScore"):
    
    _, _, F1 = bertscore(
        [pred],
        [ref],
        lang="en",
        model_type="distilbert-base-uncased",
        device=device
    )

    bert_f1_list.append(float(F1[0]))

mean_f1 = sum(bert_f1_list) / len(bert_f1_list)

print("\nFinal Mean BERTScore F1:", mean_f1)


Using device: cuda


DistilBERT BERTScore:   0%|          | 0/150 [00:00<?, ?it/s]


Final Mean BERTScore F1: 0.756071925163269
